In [12]:
import os
import pandas as pd

from scripts.utils import get_mend_df, rep_mend_analysis, mend_f1_table, get_thresh_score, add_cm_col, export_cm_examples
from src.utils import load_env, set_seed,load_json, get_logger
from src.experiment_config import ExperimentConfig

In [13]:
env_vars = load_env()
seed = env_vars["RANDOM_SEED"]
set_seed(seed)

# LOAD CONFIG FOR DATASET OF INTEREST USING ROB RESULTS CONFIG (mendelsohn labelled tweets)
rob_4c_path = "/Users/alexleto/projects/metaphor-detector/results/metaphor_filter/rule_based/tweets_immigration_metaphor_paths_scores.json"
rb_results = load_json(rob_4c_path)

# load experiment config, update paths
config = ExperimentConfig.from_dict(rb_results["config"], logger=get_logger("eval"))
config.repo_dir = env_vars["REPO_PATH"]  # reset repo dir
config.raw_data_dir = env_vars["RAW_DATA_DIR"]  # reset raw data dir
config.db_dir = env_vars["DB_DIR"]  # reset db dir

In [14]:
# load the rule-based, roberta 4 class results
rob4_results = rb_results["data"]
rob4_dicts = [{"doc_id": int(r["sdp"][0]["token_id"].split("_")[0]), "sent_id": r["sdp"][0]["token_id"].split("_")[0], \
               "matches": [item["path"] for item in r["met_paths"]], \
                "met_scores": [item["metaphor_score"] for item in r["met_paths"]], \
                "tokens": [sdp_tok["text"] for sdp_tok in r["sdp"]], \
                  "sentence_text": r["sentence_text"]} for r in rob4_results]
rob4_df = pd.DataFrame.from_dict(rob4_dicts)
# rob4_df.head()

In [15]:
exploded_df = rob4_df.explode(column=["matches", "met_scores"])
exploded_df

,doc_id,sent_id,matches,met_scores,tokens,sentence_text
0,1207888081698852864,1207888081698852864,0,3,"[None, of, canidates]",None of these canidates do.
1,1128648413900419074,1128648413900419074,2,0,"[I, look]",I look forward to learning how that’s not oppo...
2,1128648413900419074,1128648413900419074,4,0,"[opportunistic, racism]",I look forward to learning how that’s not oppo...
3,1062154241530388481,1062154241530388481,3,0,"[United, States]",Deployed Inside the United States: The Militar...
3,1062154241530388481,1062154241530388481,8,0,"[United, States]",Deployed Inside the United States: The Militar...
...,...,...,...,...,...,...
15750,1114761926893031424,1114761926893031424,5,0,"[classrooms, are, &amp]",They cause hit &amp; runs on a weekly basis &a...
15751,1114761926893031424,1114761926893031424,4,0,"[huge, &amp]",They cause hit &amp; runs on a weekly basis &a...
15752,1114761926893031424,1114761926893031424,0,0,"[half, of, them]",They cause hit &amp; runs on a weekly basis &a...
15753,1114761926893031424,1114761926893031424,2,0,"[half, speak]",They cause hit &amp; runs on a weekly basis &a...


In [16]:
grouped_df = exploded_df.groupby("matches").agg(count = ("sent_id", "count"))
grouped_df # missing 0 and 6

,count
matches,
0,1515
1,3134
2,3385
3,2798
4,2522
5,428
6,98
7,195
8,2798


In [19]:
# sort by met score and output
sorted_df = exploded_df.sort_values(by= "met_scores")
sorted_df = sorted_df[["met_scores", "tokens", "sentence_text"]]
sorted_df.to_csv("temp.csv")